# 02 数据清洗与分析底座

这一阶段的目标不是急着做图，而是把原始 CSV 转成可靠的分析层。

本项目把数据分成三层：

- `raw`：原始数据，只读不改
- `processed`：清洗、去重、聚合后的分析数据
- `reports` / `dashboard`：面向展示的结果

本 Notebook 会生成一个核心文件：`orders_analysis_base.csv`。它是一张订单粒度宽表，后续 EDA、SQL、Dashboard、机器学习都会优先基于它。

## 0. 导入依赖与路径


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

current_dir = Path.cwd()
project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir
raw_dir = project_root / "data" / "raw"
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

raw_dir, processed_dir


## 1. 读取原始数据

先把 9 张表读入内存。这里不做任何原地修改，所有清洗都生成新的 DataFrame。

In [ ]:
customers = pd.read_csv(raw_dir / "olist_customers_dataset.csv")
geolocation = pd.read_csv(raw_dir / "olist_geolocation_dataset.csv")
items = pd.read_csv(raw_dir / "olist_order_items_dataset.csv")
payments = pd.read_csv(raw_dir / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(raw_dir / "olist_order_reviews_dataset.csv")
orders = pd.read_csv(raw_dir / "olist_orders_dataset.csv")
products = pd.read_csv(raw_dir / "olist_products_dataset.csv")
sellers = pd.read_csv(raw_dir / "olist_sellers_dataset.csv")
category_translation = pd.read_csv(raw_dir / "product_category_name_translation.csv")

table_shapes = pd.DataFrame({
    "table": ["customers", "geolocation", "items", "payments", "reviews", "orders", "products", "sellers", "category_translation"],
    "rows": [len(customers), len(geolocation), len(items), len(payments), len(reviews), len(orders), len(products), len(sellers), len(category_translation)],
    "columns": [customers.shape[1], geolocation.shape[1], items.shape[1], payments.shape[1], reviews.shape[1], orders.shape[1], products.shape[1], sellers.shape[1], category_translation.shape[1]],
})
table_shapes


## 2. 清洗订单表

`orders` 是中心事实表。这里先把时间字段转换成 datetime，然后派生履约相关指标。

注意：有些订单没有完成配送，所以实际送达时间缺失是合理的业务缺失，不应该盲目删除。

In [ ]:
orders_clean = orders.copy()

date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in date_cols:
    orders_clean[col] = pd.to_datetime(orders_clean[col], errors="coerce")

orders_clean["purchase_date"] = orders_clean["order_purchase_timestamp"].dt.date
orders_clean["purchase_month"] = orders_clean["order_purchase_timestamp"].dt.to_period("M").astype(str)
orders_clean["purchase_year"] = orders_clean["order_purchase_timestamp"].dt.year
orders_clean["purchase_dayofweek"] = orders_clean["order_purchase_timestamp"].dt.day_name()

orders_clean["is_delivered"] = orders_clean["order_status"].eq("delivered")
orders_clean["approval_time_hours"] = (
    orders_clean["order_approved_at"] - orders_clean["order_purchase_timestamp"]
).dt.total_seconds() / 3600
orders_clean["delivery_days"] = (
    orders_clean["order_delivered_customer_date"] - orders_clean["order_purchase_timestamp"]
).dt.total_seconds() / 86400
orders_clean["estimated_delivery_days"] = (
    orders_clean["order_estimated_delivery_date"] - orders_clean["order_purchase_timestamp"]
).dt.total_seconds() / 86400
orders_clean["delivery_delta_days"] = (
    orders_clean["order_delivered_customer_date"] - orders_clean["order_estimated_delivery_date"]
).dt.total_seconds() / 86400
orders_clean["is_late"] = orders_clean["delivery_delta_days"] > 0

orders_clean[["order_id", "order_status", "purchase_month", "delivery_days", "estimated_delivery_days", "delivery_delta_days", "is_late"]].head()


## 3. 清洗商品表

原始字段里有 `lenght` 拼写问题。我在 processed 层修正字段名，同时保留 raw 层不动。

同时把葡萄牙语品类名映射成英文，方便后续展示。

In [ ]:
products_clean = products.copy().rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length",
})

products_clean = products_clean.merge(
    category_translation,
    on="product_category_name",
    how="left",
)

products_clean["product_category_name_english"] = products_clean["product_category_name_english"].fillna("unknown")
products_clean["product_category_name"] = products_clean["product_category_name"].fillna("unknown")

products_clean.head()


## 4. 清洗地理位置表

`geolocation` 是最容易误用的一张表：同一个邮编前缀会出现很多条经纬度记录。直接 join 会把订单行数放大。

处理方式：先压缩成每个 zip code prefix 一行，再和客户/卖家表连接。经纬度用中位数，城市和州用出现频率最高的值。

In [ ]:
def mode_or_first(series: pd.Series):
    mode = series.dropna().mode()
    if len(mode) > 0:
        return mode.iloc[0]
    return series.dropna().iloc[0] if series.dropna().size else np.nan

geolocation_clean = (
    geolocation
    .groupby("geolocation_zip_code_prefix", as_index=False)
    .agg(
        geolocation_lat=("geolocation_lat", "median"),
        geolocation_lng=("geolocation_lng", "median"),
        geolocation_city=("geolocation_city", mode_or_first),
        geolocation_state=("geolocation_state", mode_or_first),
        geolocation_records=("geolocation_zip_code_prefix", "size"),
    )
)

print("raw geolocation rows:", len(geolocation))
print("clean geolocation rows:", len(geolocation_clean))
geolocation_clean.head()


## 5. 聚合支付表

一个订单可能有多笔支付记录，所以要先聚合到订单粒度。这里保留支付总额、支付次数、最大分期数、主要支付方式。

In [ ]:
payment_type_value = (
    payments
    .groupby(["order_id", "payment_type"], as_index=False)["payment_value"]
    .sum()
)

primary_payment_type = (
    payment_type_value
    .sort_values(["order_id", "payment_value"], ascending=[True, False])
    .drop_duplicates("order_id")
    .rename(columns={"payment_type": "primary_payment_type"})[["order_id", "primary_payment_type"]]
)

payments_agg = (
    payments
    .groupby("order_id", as_index=False)
    .agg(
        payment_total=("payment_value", "sum"),
        payment_count=("payment_sequential", "max"),
        payment_type_count=("payment_type", "nunique"),
        max_payment_installments=("payment_installments", "max"),
    )
    .merge(primary_payment_type, on="order_id", how="left")
)

payments_agg.head()


## 6. 聚合评价表

一个订单理论上主要对应一条评价，但实际数据中可能存在重复或多条记录。这里聚合到订单粒度，并派生是否有文字评论、是否低评分。

In [ ]:
reviews_clean = reviews.copy()
reviews_clean["review_creation_date"] = pd.to_datetime(reviews_clean["review_creation_date"], errors="coerce")
reviews_clean["review_answer_timestamp"] = pd.to_datetime(reviews_clean["review_answer_timestamp"], errors="coerce")
reviews_clean["has_review_title"] = reviews_clean["review_comment_title"].notna()
reviews_clean["has_review_message"] = reviews_clean["review_comment_message"].notna()

reviews_agg = (
    reviews_clean
    .groupby("order_id", as_index=False)
    .agg(
        review_score_mean=("review_score", "mean"),
        review_score_min=("review_score", "min"),
        review_score_max=("review_score", "max"),
        review_count=("review_id", "count"),
        has_review_title=("has_review_title", "max"),
        has_review_message=("has_review_message", "max"),
    )
)
reviews_agg["is_low_review"] = reviews_agg["review_score_min"] <= 2

reviews_agg.head()


## 7. 构建商品明细增强表

`items` 是订单商品明细表，一笔订单可能有多个商品，也可能涉及多个卖家。这里先把商品和卖家信息接上，再形成 item-level 明细。

In [ ]:
items_enriched = (
    items
    .merge(products_clean, on="product_id", how="left")
    .merge(sellers, on="seller_id", how="left")
)

items_enriched["shipping_limit_date"] = pd.to_datetime(items_enriched["shipping_limit_date"], errors="coerce")
items_enriched["item_total"] = items_enriched["price"] + items_enriched["freight_value"]
items_enriched["freight_ratio"] = np.where(
    items_enriched["price"] > 0,
    items_enriched["freight_value"] / items_enriched["price"],
    np.nan,
)

items_enriched.head()


## 8. 聚合商品明细到订单粒度

后续做月度收入、订单履约、评价分析时，最常用的是一行一订单。所以需要把 item-level 明细聚合到 order-level。

In [ ]:
def top_value_by_revenue(group: pd.DataFrame, value_col: str) -> object:
    revenue = group.groupby(value_col, dropna=False)["price"].sum().sort_values(ascending=False)
    return revenue.index[0] if len(revenue) else np.nan

main_category = (
    items_enriched
    .groupby("order_id")
    .apply(lambda g: top_value_by_revenue(g, "product_category_name_english"), include_groups=False)
    .reset_index(name="main_product_category")
)

main_seller_state = (
    items_enriched
    .groupby("order_id")
    .apply(lambda g: top_value_by_revenue(g, "seller_state"), include_groups=False)
    .reset_index(name="main_seller_state")
)

items_agg = (
    items_enriched
    .groupby("order_id", as_index=False)
    .agg(
        item_count=("order_item_id", "count"),
        product_count=("product_id", "nunique"),
        seller_count=("seller_id", "nunique"),
        product_category_count=("product_category_name_english", "nunique"),
        product_total=("price", "sum"),
        freight_total=("freight_value", "sum"),
        item_total=("item_total", "sum"),
        avg_item_price=("price", "mean"),
        avg_freight_ratio=("freight_ratio", "mean"),
    )
    .merge(main_category, on="order_id", how="left")
    .merge(main_seller_state, on="order_id", how="left")
)

items_agg.head()


## 9. 构建订单粒度分析宽表

这是本阶段最重要的产出：一行代表一个订单。

宽表不是为了取代原始明细表，而是为了让常见业务分析更稳定、更高效。后面如果要做商品明细分析，仍然可以回到 `items_enriched`。

In [ ]:
customers_geo = customers.merge(
    geolocation_clean.add_prefix("customer_"),
    left_on="customer_zip_code_prefix",
    right_on="customer_geolocation_zip_code_prefix",
    how="left",
)

orders_analysis_base = (
    orders_clean
    .merge(customers_geo, on="customer_id", how="left")
    .merge(items_agg, on="order_id", how="left")
    .merge(payments_agg, on="order_id", how="left")
    .merge(reviews_agg, on="order_id", how="left")
)

orders_analysis_base["freight_share_of_payment"] = np.where(
    orders_analysis_base["payment_total"] > 0,
    orders_analysis_base["freight_total"] / orders_analysis_base["payment_total"],
    np.nan,
)

orders_analysis_base["has_review"] = orders_analysis_base["review_count"].notna()

orders_analysis_base.shape


## 10. 质量检查

清洗后需要做质量检查。尤其是 join 之后，要确认没有因为一对多连接导致订单数膨胀。

In [ ]:
assert len(orders_analysis_base) == len(orders_clean), "订单宽表行数应该等于 orders 表行数"
assert orders_analysis_base["order_id"].is_unique, "订单宽表必须保持 order_id 唯一"

quality_summary = pd.DataFrame({
    "check": [
        "orders_raw_rows",
        "orders_analysis_base_rows",
        "unique_order_ids",
        "delivered_orders",
        "orders_with_items",
        "orders_with_payments",
        "orders_with_reviews",
        "late_delivered_orders",
    ],
    "value": [
        len(orders),
        len(orders_analysis_base),
        orders_analysis_base["order_id"].nunique(),
        int(orders_analysis_base["is_delivered"].sum()),
        int(orders_analysis_base["item_count"].notna().sum()),
        int(orders_analysis_base["payment_total"].notna().sum()),
        int(orders_analysis_base["review_count"].notna().sum()),
        int((orders_analysis_base["is_delivered"] & orders_analysis_base["is_late"]).sum()),
    ],
})

quality_summary


In [ ]:
missing_summary = (
    orders_analysis_base
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"index": "column", 0: "missing_pct"})
)

missing_summary.head(20)


## 11. 保存 processed 层数据

保存这些文件后，后续 Notebook 不需要每次重复清洗逻辑，可以直接读取 processed 数据。

In [ ]:
orders_clean.to_csv(processed_dir / "orders_clean.csv", index=False)
products_clean.to_csv(processed_dir / "products_clean.csv", index=False)
geolocation_clean.to_csv(processed_dir / "geolocation_clean.csv", index=False)
payments_agg.to_csv(processed_dir / "order_payments_agg.csv", index=False)
reviews_agg.to_csv(processed_dir / "order_reviews_agg.csv", index=False)
items_enriched.to_csv(processed_dir / "order_items_enriched.csv", index=False)
items_agg.to_csv(processed_dir / "order_items_agg.csv", index=False)
orders_analysis_base.to_csv(processed_dir / "orders_analysis_base.csv", index=False)
quality_summary.to_csv(processed_dir / "data_quality_summary.csv", index=False)
missing_summary.to_csv(processed_dir / "orders_analysis_missing_summary.csv", index=False)

sorted(path.name for path in processed_dir.glob("*.csv"))


## 12. 本阶段结论

本阶段完成了项目的关键数据底座：

- raw 层数据保持不变
- geolocation 先去重聚合，避免错误 join
- payments / reviews / items 统一聚合到订单粒度
- 构建出 `orders_analysis_base.csv` 作为后续分析主表
- 用 assert 检查宽表没有行数膨胀

下一步是 `03_eda_business_analysis.ipynb`：围绕 GMV、订单量、客单价、物流延迟、评价、品类、地区建立商业分析框架。